## Load Model and cache

In [1]:
import numpy as np

Zb_te   = np.load("cache_Zb_te.npy")
Za_te   = np.load("cache_Za_te.npy")
Zb_te3  = np.load("cache_Zb_te3.npy")
Za_te3  = np.load("cache_Za_te3.npy")

pred_b  = np.load("cache_pred_b.npy")
pred_a  = np.load("cache_pred_a.npy")

err_b   = np.load("cache_err_b.npy")
err_a   = np.load("cache_err_a.npy")

phi_test_n = np.load("cache_phi_test_n.npy")

hi_idx = np.load("cache_hi_idx.npy")
lo_idx = np.load("cache_lo_idx.npy")

print("Reloaded cached arrays")
print("Zb_te:", Zb_te.shape, "Za_te:", Za_te.shape)
print("Zb_te3:", Zb_te3.shape, "Za_te3:", Za_te3.shape)
print("pred:", pred_b.shape, pred_a.shape)
print("err:", err_b.shape, err_a.shape)
print("phi_test_n:", phi_test_n.shape)
print("hi/lo:", hi_idx.shape, lo_idx.shape)
print("Mean abs error baseline vs aug:", err_b.mean(), err_a.mean())


Reloaded cached arrays
Zb_te: (2742, 256) Za_te: (2742, 256)
Zb_te3: (2742, 3) Za_te3: (2742, 3)
pred: (2742,) (2742,)
err: (2742,) (2742,)
phi_test_n: (2742, 3)
hi/lo: (27,) (27,)
Mean abs error baseline vs aug: 0.004530530015382082 0.0038683008783624603


In [3]:
#3b - Models


import torch.nn as nn

class BiLSTMBase(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)          # (B, T, 2H)
        h_last = out[:, -1, :]         # (B, 2H)
        return self.head(h_last)       # (B, 1)

class BiLSTMTopoAug(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, num_layers=1, topo_dim=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2 + topo_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x, phi):
        out, _ = self.lstm(x)
        h_last = out[:, -1, :]
        z = torch.cat([h_last, phi], dim=1)
        return self.head(z)


In [4]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt_b = torch.load("saved_models/bilstm_base_best.pt", map_location=device, weights_only=False)
ckpt_a = torch.load("saved_models/bilstm_topo_aug_best.pt", map_location=device, weights_only=False)

base_best = BiLSTMBase(
    input_dim=ckpt_b["config"]["input_dim"],
    hidden_dim=ckpt_b["config"]["hidden_dim"]
).to(device)
base_best.load_state_dict(ckpt_b["model_state"])
base_best.eval()

aug_best = BiLSTMTopoAug(
    input_dim=ckpt_a["config"]["input_dim"],
    hidden_dim=ckpt_a["config"]["hidden_dim"],
    topo_dim=ckpt_a["config"]["topo_dim"]
).to(device)
aug_best.load_state_dict(ckpt_a["model_state"])
aug_best.eval()

print("Reloaded best models")
print("baseline rmse saved:", ckpt_b.get("rmse", None))
print("topo-aug rmse saved:", ckpt_a.get("rmse", None))


Reloaded best models
baseline rmse saved: 0.0072070877
topo-aug rmse saved: 0.0056256014


In [5]:
import numpy as np

W = aug_best.head[0].weight.detach().cpu().numpy()   # first Linear layer in head
topo_dim = phi_test_n.shape[1]

W_lstm = W[:, :-topo_dim]      # weights connected to BiLSTM latent
W_topo = W[:, -topo_dim:]      # weights connected to zigzag features

print("Topo dim:", topo_dim)
print("Mean |weight| on LSTM inputs:", np.mean(np.abs(W_lstm)))
print("Mean |weight| on TOPO inputs:", np.mean(np.abs(W_topo)))
print("Mean |weight| per topo feature:", np.mean(np.abs(W_topo), axis=0))
print("Max  |weight| per topo feature:", np.max(np.abs(W_topo), axis=0))


Topo dim: 3
Mean |weight| on LSTM inputs: 0.04479879
Mean |weight| on TOPO inputs: 0.030637475
Mean |weight| per topo feature: [0.03139887 0.03091197 0.02960156]
Max  |weight| per topo feature: [0.09825325 0.08331489 0.1038376 ]


In [6]:
# permutation test 
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

CSV_PATH = r"/home/thesis/Downloads/Thesis-20251203T130544Z-1-001/Thesis/MSEA/MSEA-Res database v2/MSEA-Res database v2/Dynamic_components/Level2/Level2_5143_1973_Nam_Oun.csv"

df = pd.read_csv(CSV_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").dropna(subset=["Storage_km3"]).reset_index(drop=True)

print("Data shape:", df.shape)
print(df.head())
print(df.tail())


Data shape: (13180, 2)
        Date  Storage_km3
0 1987-12-01     0.367029
1 1987-12-02     0.367728
2 1987-12-03     0.368428
3 1987-12-04     0.369127
4 1987-12-05     0.369826
            Date  Storage_km3
13175 2023-12-27     0.380401
13176 2023-12-28     0.380172
13177 2023-12-29     0.379965
13178 2023-12-30     0.379781
13179 2023-12-31     0.379621


In [7]:

# ----------------------------
# 2) Temporal split (train past, test future)
# ----------------------------
cutoff = pd.Timestamp("2016-01-01")  # same idea as before; change if you want
train_df = df[df["Date"] < cutoff].copy()
test_df  = df[df["Date"] >= cutoff].copy()

print("\nTrain:", train_df["Date"].min(), "→", train_df["Date"].max(), "n =", len(train_df))
print("Test :", test_df["Date"].min(),  "→", test_df["Date"].max(),  "n =", len(test_df))



Train: 1987-12-01 00:00:00 → 2015-12-31 00:00:00 n = 10258
Test : 2016-01-01 00:00:00 → 2023-12-31 00:00:00 n = 2922


In [8]:

# ----------------------------
# 3) Scale using TRAIN only
# ----------------------------
scaler = StandardScaler().fit(train_df[["Storage_km3"]])
train_s = scaler.transform(train_df[["Storage_km3"]]).flatten()
test_s  = scaler.transform(test_df[["Storage_km3"]]).flatten()

# ----------------------------
# 4) Make supervised windows
# ----------------------------
def make_windows(series, lookback=180, horizon=1, stride=1):
    """
    series: 1D array
    returns:
      X: [N, lookback, 1]
      y: [N] if horizon=1 else [N, horizon]
    """
    X, y = [], []
    for i in range(0, len(series) - lookback - horizon + 1, stride):
        X.append(series[i:i+lookback])
        y.append(series[i+lookback:i+lookback+horizon])
    X = np.asarray(X)[:, :, None]
    y = np.asarray(y).squeeze()
    return X, y

LOOKBACK = 180
HORIZON  = 1

X_train, y_train = make_windows(train_s, lookback=LOOKBACK, horizon=HORIZON)
X_test,  y_test  = make_windows(test_s,  lookback=LOOKBACK, horizon=HORIZON)

print("\nWindowed shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)


Windowed shapes:
X_train: (10078, 180, 1) y_train: (10078,)
X_test : (2742, 180, 1) y_test : (2742,)


## Implementation of RMSE Evaluation

In [9]:
import numpy as np
import torch

@torch.no_grad()
def rmse_aug(model, X_np, phi_np, y_np, device, batch_size=256):
    model.eval()
    X = torch.tensor(X_np, dtype=torch.float32)
    P = torch.tensor(phi_np, dtype=torch.float32)
    y = torch.tensor(y_np, dtype=torch.float32)

    preds = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size].to(device)
        pb = P[i:i+batch_size].to(device)
        yhat = model(xb, pb).detach().cpu().numpy()
        preds.append(yhat)

    pred = np.concatenate(preds)
    return float(np.sqrt(np.mean((pred - y_np)**2)))

# 1) True (unshuffled) RMSE
rmse_true = rmse_aug(aug_best, X_test, phi_test_n, y_test, device)
print("RMSE (true phi):", rmse_true)

# 2) Permutation RMSEs
rng = np.random.default_rng(0)
n_perm = 20
rmse_perm = []

for k in range(n_perm):
    perm = rng.permutation(len(phi_test_n))
    phi_shuf = phi_test_n[perm]
    rm = rmse_aug(aug_best, X_test, phi_shuf, y_test, device)
    rmse_perm.append(rm)

rmse_perm = np.array(rmse_perm)
print("RMSE (shuffled phi) mean±std:", rmse_perm.mean(), rmse_perm.std())
print("RMSE increase (mean):", rmse_perm.mean() - rmse_true)
print("All shuffled RMSEs:", rmse_perm)


RMSE (true phi): 1.386391944576292
RMSE (shuffled phi) mean±std: 1.3856760450982206 5.145316090380427e-05
RMSE increase (mean): -0.0007158994780713801
All shuffled RMSEs: [1.38560805 1.38569395 1.38568693 1.38569286 1.38576864 1.38561396
 1.38564611 1.38553821 1.38565369 1.38567903 1.38565986 1.38571784
 1.38570271 1.38576725 1.38572567 1.38567439 1.38565764 1.38565807
 1.38567531 1.38570071]


In [10]:
rmse_drop = []
for j in range(phi_test_n.shape[1]):
    phi_one = phi_test_n.copy()
    rng = np.random.default_rng(123 + j)
    phi_one[:, j] = phi_one[rng.permutation(len(phi_one)), j]  # shuffle only feature j
    rm = rmse_aug(aug_best, X_test, phi_one, y_test, device)
    rmse_drop.append(rm)

rmse_drop = np.array(rmse_drop)
print("RMSE true:", rmse_true)
print("RMSE when shuffling each topo feature:", rmse_drop)
print("Increase per topo feature:", rmse_drop - rmse_true)


RMSE true: 1.386391944576292
RMSE when shuffling each topo feature: [1.38626447 1.38600673 1.38461932]
Increase per topo feature: [-0.00012747 -0.00038521 -0.00177263]
